# Phase 1 EDA — Cleaned Reviews

Explores `data/processed/reviews_clean.parquet` produced by `src/preprocessing/clean_text.py`.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
import config

sns.set_theme(style="whitegrid")

df = pd.read_parquet(config.PROCESSED_DIR / "reviews_clean.parquet")
df.head()

## Null / duplicate check

In [ ]:
print("Rows:", len(df))
print("\nNull counts:")
print(df.isnull().sum())

n_dupes = df.duplicated(subset=[config.REVIEW_TEXT_COL]).sum()
print(f"\nDuplicate reviews (by raw text): {n_dupes}")

**Takeaway:** confirms the parquet is clean going into analysis — no nulls in the text/cleaning columns, and any duplicate reviews are visible so we can decide whether to drop them before modeling.

## Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=config.RATING_COL, data=df, order=sorted(df[config.RATING_COL].unique()), ax=ax)
ax.set_title("Rating Distribution")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

**Takeaway:** shows whether the sample skews toward positive or negative reviews, which matters for interpreting sentiment scores later — a skewed distribution means baseline sentiment metrics should be read relative to this skew, not assumed neutral.

## Review length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(df["word_count"], bins=20, ax=ax)
ax.set_title("Review Length Distribution (word count)")
ax.set_xlabel("Word Count")
ax.set_ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

**Takeaway:** most reviews are short; a long tail of longer reviews likely carries more detailed complaints/praise and is worth weighting more heavily in aspect mining.

## Top 20 most frequent words (clean_deep)

In [ ]:
all_words = " ".join(df["clean_deep"].dropna()).split()
word_counts = Counter(all_words)
top_words = word_counts.most_common(20)
top_df = pd.DataFrame(top_words, columns=["word", "count"])

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x="count", y="word", data=top_df, ax=ax)
ax.set_title("Top 20 Most Frequent Words (clean_deep)")
ax.set_xlabel("Frequency")
ax.set_ylabel("Word")
plt.tight_layout()
plt.show()

**Takeaway:** the most frequent lemmas hint at the dominant themes (e.g. delivery, quality, support, price) that later aspect-mining phases should expect to surface as topics.

## Word cloud (clean_deep)

In [ ]:
from wordcloud import WordCloud

text_blob = " ".join(df["clean_deep"].dropna())
wc = WordCloud(width=800, height=400, background_color="white").generate(text_blob)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — clean_deep")
plt.tight_layout()
plt.show()

**Takeaway:** a quick visual gut-check of vocabulary — larger words should match the top-20 bar chart above; any surprise dominant term may indicate a cleaning issue (e.g. a stopword that slipped through).

## Sanity check: average word count by rating

In [ ]:
avg_wc_by_rating = df.groupby(config.RATING_COL)["word_count"].mean().round(1)
print(avg_wc_by_rating)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=avg_wc_by_rating.index, y=avg_wc_by_rating.values, ax=ax)
ax.set_title("Average Word Count by Rating")
ax.set_xlabel("Rating")
ax.set_ylabel("Average Word Count")
plt.tight_layout()
plt.show()

**Takeaway:** if low/high ratings have noticeably longer reviews than mid ratings, that's a common VOC pattern (people write more when strongly satisfied or dissatisfied) worth keeping in mind when weighting reviews in later phases.